# 7. Streamlit and FastAPI interfaces

**Learning objective:** see how the same domain service is exposed twice, once as
a local page for a person and once as HTTP for a future front end, without either
interface owning any domain logic.

**Where this fits:** everything below the dotted line was built in notebooks 1
through 6 and is unchanged here.

```
Streamlit page ---\
                   >--- MosaicPathwayService --- retrieval + generation
FastAPI service --/
.........................................................................
```

No server is started from this notebook. The API is exercised in process with
`TestClient`, and the Streamlit helpers are pure functions that need no browser.

## Presentation and transport are thin on purpose

The rule this project follows is that an interface may parse, validate, present,
and map errors. It may not retrieve, generate, chunk, score, or decide anything
about a pathway.

That is why the Streamlit app is split in two. `app.py` holds the widgets and the
layout, which cannot be tested without a browser. `app_support.py` holds the
parsing, the validation, the session state handling, and the previews, which are
ordinary functions with ordinary tests.

The API follows the same split: `create_app` builds routes and error mappings,
and the service is injected.

In [ ]:
from contextlib import nullcontext

from fastapi.testclient import TestClient

from mosaic_pathway.api import (
    PREVIEW_CHARACTERS,
    create_app,
    parse_allowed_origins,
    summarize_sources,
)
from mosaic_pathway.app_support import (
    ChildInput,
    IntakeFormInput,
    build_family_intake,
    clear_pathway,
    describe_generation_failure,
    evidence_preview,
    parse_comma_separated,
    read_output_view,
    store_error,
    store_result,
    validate_form_input,
)
from mosaic_pathway.models import (
    CommunitySuggestion,
    FamilyIntake,
    GroundedPathwayResult,
    LearningPathway,
    ResourceRecommendation,
    RetrievedRecord,
    RhythmPractice,
    SourceRecord,
)
from mosaic_pathway.rag import GroundingError

print("preview limit shared by both interfaces:", PREVIEW_CHARACTERS)

## Parsing what a person types

A text box gives you a string. The domain wants a list. In between sit trailing
commas, double spaces, and the same interest typed twice in different cases.

`parse_comma_separated` handles all of it in one place, so the same rules apply
to every field on the form.

In [ ]:
print(parse_comma_separated("animals, drawing , , Animals,  reading "))
print(parse_comma_separated(""))

## Two layers of validation

Some rules cannot be expressed in the domain model. "At least one goal field is
filled in" spans six form fields and has no meaning inside `FamilyIntake`, so it
lives in `validate_form_input` and runs before the model is built.

The domain model still validates afterwards. The form check produces friendly
guidance; the model check is the boundary that cannot be bypassed.

In [ ]:
empty_form = IntakeFormInput(children=[ChildInput(label="", age=9)])

for message in validate_form_input(empty_form):
    print("-", message)

In [ ]:
form = IntakeFormInput(
    children=[
        ChildInput(
            label="older child",
            age=12,
            interests="animals, drawing",
            learning_needs="movement breaks",
        )
    ],
    leaving_behind="rigid daily schedules",
    wants_to_preserve="reading together after dinner",
    wants_to_add="more time outdoors",
    family_values="curiosity, gentleness",
    practical_constraints="one working parent at home",
    additional_context="  ",
)

print("form problems:", validate_form_input(form))

intake = build_family_intake(form)

print("children      :", [child.label for child in intake.children])
print("interests     :", intake.children[0].interests)
print("values        :", intake.family_values)
print("blank context become None:", intake.additional_context is None)

## Session state without Streamlit

Streamlit reruns the whole script on every interaction, so anything that must
survive a click has to live in `st.session_state`. The helpers treat that store
as a plain mutable mapping, which means a dictionary works here and in tests.

The rules encoded in those helpers are small but they are product decisions: a
new failure never erases the pathway a family is already reading, and a new
success clears the previous error.

In [ ]:
def make_record(index: int, text: str) -> SourceRecord:
    return SourceRecord(
        source_id=f"synthetic-guide-{index:04d}",
        title=f"Synthetic guide chunk {index}",
        source_file="synthetic-guide.docx",
        content_type="practical_guidance",
        authority_type="mosaic_guidance",
        topics=["rhythm"],
        text=text,
    )


def make_pathway() -> LearningPathway:
    return LearningPathway(
        family_reflection="Your family is making room for animals, drawing, and slower mornings.",
        starting_rhythm=[
            RhythmPractice(
                timing="Most mornings",
                practice="Take a short walk before the day begins.",
                why_it_fits="It adds outdoor time without adding a schedule.",
            ),
            RhythmPractice(
                timing="Once a week",
                practice="Add one page to a shared observation journal.",
                why_it_fits="It keeps a record of what your child noticed.",
            ),
        ],
        resources=[
            ResourceRecommendation(
                title="Start an interest catalog",
                why_it_fits="It gives a growing interest somewhere to live.",
                source_id="synthetic-guide-0007",
                url=None,
            ),
            ResourceRecommendation(
                title="Build a gentle weekly rhythm",
                why_it_fits="It replaces the timetable you are leaving behind.",
                source_id="synthetic-guide-0011",
                url=None,
            ),
        ],
        community_suggestion=CommunitySuggestion(
            suggestion="Visit one informal nature meetup this month.",
            why_it_fits="It is a low pressure way to meet other families.",
            source_id="synthetic-guide-0007",
        ),
        closing_note="Go slowly. One walk and one journal page is a real start.",
    )


LONG_TEXT = "A synthetic passage about gentle weekly rhythms for families. " * 8
TAIL_MARKER = "SYNTHETIC-TAIL-MARKER"


def make_result(intake: FamilyIntake) -> GroundedPathwayResult:
    return GroundedPathwayResult(
        intake=intake,
        retrieval_query="synthetic retrieval query",
        retrieved_records=[
            RetrievedRecord(
                record=make_record(7, LONG_TEXT + TAIL_MARKER), score=0.8123456
            ),
            RetrievedRecord(
                record=make_record(
                    11, "A shorter synthetic passage about interest catalogs."
                ),
                score=0.7,
            ),
        ],
        pathway=make_pathway(),
    )


print("passage length:", len(LONG_TEXT + TAIL_MARKER), "characters")

In [ ]:
state: dict[str, object] = {}

print("first render shows placeholder:", read_output_view(state).show_placeholder)

store_result(state, make_result(intake))
print("after success, placeholder:", read_output_view(state).show_placeholder)

failure = GroundingError("cited synthetic-guide-0042")
store_error(state, describe_generation_failure(failure), failure)
view = read_output_view(state)

print("after a later failure, previous pathway kept:", view.shows_previous_result)
print("family-facing message:", view.error.message if view.error else None)
print("technical detail kept separate:", view.error.detail if view.error else None)

clear_pathway(state)
print("after clearing, placeholder:", read_output_view(state).show_placeholder)

Notice the failure message. `describe_generation_failure` maps an exception type
onto one sentence a family can act on, and keeps the technical detail in a
separate field that the page hides behind an expander.

## The API: a factory, not a module-level app

`create_app` is an application factory. It takes a service provider and the
allowed CORS origins, and returns a configured `FastAPI` instance. The module
still exposes `app = create_app()` for `uvicorn`, but nothing forces a test or a
notebook to use that one.

The provider is a context manager rather than a plain callable, because the real
service owns a Qdrant handle that must be closed. It is entered once during
startup and exited during shutdown, so the embedding model and the collection are
loaded once for the process instead of once per request.

If startup fails, the error is captured rather than raised. `/health` must stay
answerable so an operator can tell the difference between "not running" and
"running but misconfigured".

In [ ]:
class FakePathwayService:
    """Returns a fixed grounded result, or raises, without any real dependency."""

    def __init__(self, error: Exception | None = None) -> None:
        self.error = error
        self.calls: list[FamilyIntake] = []

    def generate_pathway(self, intake: FamilyIntake) -> GroundedPathwayResult:
        self.calls.append(intake)

        if self.error is not None:
            raise self.error

        return make_result(intake)


def build_client(service: FakePathwayService, origins: list[str]) -> TestClient:
    return TestClient(
        create_app(
            service_provider=lambda: nullcontext(service), allowed_origins=origins
        )
    )


fake_service = FakePathwayService()
client = build_client(fake_service, ["http://localhost:5173"])

print("client built against a fake service")

In [ ]:
with client:
    health = client.get("/health")

print(health.status_code, health.json())
print("service was not touched:", fake_service.calls == [])

## The success path over HTTP

The request body is the `FamilyIntake` model from notebook 1, so the HTTP
contract and the domain contract are the same object. FastAPI validates the body
before the route function runs.

In [ ]:
with client:
    response = client.post("/api/v1/pathways", json=intake.model_dump())

body = response.json()

print("status:", response.status_code)
print("keys  :", sorted(body))
print("resources returned:", len(body["pathway"]["resources"]))
print()

for source in body["sources"]:
    print(
        f"{source['source_id']} | score {source['score']} | preview {len(source['preview'])} chars"
    )

## The response is a reduction, not a dump

The API never returns full source passages. `summarize_sources` keeps retrieval
rank order and reduces each record to an id, a title, a rounded score, and a
short preview.

The marker below sits well past the preview limit in the source text, so it
proves the truncation is real rather than assumed.

In [ ]:
print(
    "marker position in the source text:", (LONG_TEXT + TAIL_MARKER).index(TAIL_MARKER)
)
print("preview limit                     :", PREVIEW_CHARACTERS)
print("marker present in the API response:", TAIL_MARKER in response.text)
print()

summaries = summarize_sources(make_result(intake).retrieved_records)

for summary in summaries:
    print(
        f"{summary.source_id}: {len(summary.preview)} characters, ends with {summary.preview[-3:]!r}"
    )

print()
print(
    "the same helper backs the Streamlit evidence panel:",
    evidence_preview(LONG_TEXT + TAIL_MARKER) == summaries[0].preview,
)

## Failure paths

A malformed body never reaches the service: FastAPI rejects it with 422 and a
field-level explanation. A service failure is mapped to a stable error code so a
front end can branch on something other than a message string.

In [ ]:
calls_before = len(fake_service.calls)

with client:
    malformed = client.post("/api/v1/pathways", json={"children": []})

print("status:", malformed.status_code)
print(
    "first error:",
    malformed.json()["detail"][0]["loc"],
    malformed.json()["detail"][0]["msg"],
)
print("service was never reached:", len(fake_service.calls) == calls_before)

In [ ]:
failing_client = build_client(
    FakePathwayService(GroundingError("cited synthetic-guide-0042")),
    ["http://localhost:5173"],
)

with failing_client:
    failed = failing_client.post("/api/v1/pathways", json=intake.model_dump())

print("status:", failed.status_code)
print("detail:", failed.json()["detail"])

## CORS is configuration, not code

A browser front end on another port is a different origin. The allowed origins
are a comma-separated setting with sensible local defaults, parsed in one place.

In [ ]:
print("default origins :", parse_allowed_origins(""))
print(
    "configured value:",
    parse_allowed_origins(
        " http://localhost:5173 , http://localhost:5173 , http://localhost:4321 "
    ),
)
print()

preflight_headers = {
    "Origin": "http://localhost:5173",
    "Access-Control-Request-Method": "POST",
    "Access-Control-Request-Headers": "Content-Type",
}

with client:
    allowed = client.options("/api/v1/pathways", headers=preflight_headers)
    blocked = client.options(
        "/api/v1/pathways",
        headers=preflight_headers | {"Origin": "http://evil.test"},
    )

print(
    "allowed origin ->",
    allowed.status_code,
    allowed.headers.get("access-control-allow-origin"),
)
print(
    "other origin   ->",
    blocked.status_code,
    blocked.headers.get("access-control-allow-origin"),
)

## Known limitations of both interfaces

* The form supports one or two children, because the layout was designed for that
  and not because the domain model requires it.
* Nothing is persisted. Closing the Streamlit tab loses the pathway, and the API
  stores nothing between requests.
* The API route is synchronous. Generation blocks a worker thread, which is fine
  for local single-user use and would need revisiting under load.
* Local Qdrant allows one owning process at a time, so the Streamlit app, the
  API, and any notebook that opens the index cannot run together.
* Both interfaces are local development tools. There is no authentication, no
  rate limiting, and no deployment story.
* A React front end is a future slice. The API exists so that slice does not
  require reworking the service.

## Architecture recap

```
FamilyIntake            notebook 1: validated contracts
  |
  v
build_retrieval_query   notebook 5: deterministic query
  |
  v
MosaicRetriever         notebook 4: embeddings, local Qdrant, per-source cap
  |                     notebook 3: extraction, cleaning, chunking upstream
  v
build_context           notebook 5: the only private-text network boundary
  |
  v
ClaudePathwayGenerator  notebook 2: structured output, API key auth
  |
  v
grounding check         notebook 5: cited ids must have been retrieved
  |
  v
GroundedPathwayResult   notebook 6: deterministic checks plus human review
  |
  v
Streamlit / FastAPI     notebook 7: presentation and transport only
```

## Suggested next steps

* Read `docs/architecture.md` for the same picture in prose, and
  `docs/demo-checklist.md` for the commands that run each piece.
* Add a retrieval query to the evaluation set and watch the hit rate move.
* Add an evaluation case that you expect to fail, and see which check catches it.
* Try a different chunk size and note which ids change, and what that costs.

## Key takeaways

* One service, two interfaces, and no domain logic in either of them.
* Streamlit logic is split so that everything except the widgets is testable.
* The API is built by a factory with an injected service, which is what let this
  notebook exercise it offline.
* Responses are reductions with enforced previews, so private passages stay on
  the machine that owns them.
* The limitations above are deliberate scope, not oversights.